# Dự án Machine Learning: Cảnh báo sớm học sinh có nguy cơ điểm Toán thấp

**Nghiên cứu ứng dụng Machine Learning trên bộ dữ liệu Student Performance của UCI.**

Dự án khảo sát khả năng nhận diện sớm học sinh có nguy cơ đạt điểm cuối kỳ môn Toán dưới ngưỡng 10/20, sử dụng các thông tin học tập và bối cảnh không bao gồm điểm số G1, G2, G3. Quy trình thực nghiệm bao gồm thu thập dữ liệu, mô tả dữ liệu, khám phá dữ liệu (EDA), phân tích mối liên hệ giữa đặc trưng và nhãn, so sánh nhóm đặc trưng, kiểm tra chéo, lựa chọn mô hình/ngưỡng và đánh giá cuối cùng trên tập test độc lập.

Notebook là bản trình bày tuần tự của toàn bộ quy trình nghiên cứu. Các mô-đun trong `src/` cung cấp phiên bản mã nguồn có thể chạy từ dòng lệnh; cấu trúc và phương pháp được giữ nhất quán giữa hai hình thức.


## 1. Bộ dữ liệu và phạm vi nghiên cứu

- **Bộ dữ liệu:** *Student Performance* — tập môn Toán (`student-mat.csv`).
- **Nguồn:** UCI Machine Learning Repository, [trang dữ liệu chính thức](https://archive.ics.uci.edu/dataset/320/student-performance).
- **Tác giả được ghi nhận:** Paulo Cortez; nghiên cứu được công bố năm 2008.
- **Quy mô dữ liệu gốc:** 395 học sinh và 33 cột.
- **Bối cảnh thu thập:** hồ sơ học tập và bảng hỏi từ hai trường trung học tại Bồ Đào Nha.
- **Giấy phép:** CC BY 4.0; việc sử dụng và phân phối lại cần ghi nhận nguồn phù hợp.
- **Nhóm biến:** thông tin trường học, nhân khẩu và gia đình, hỗ trợ học tập, thói quen, số lần trượt trước đó, số buổi nghỉ học và điểm G1/G2/G3.

**Phạm vi suy luận.** Dữ liệu phản ánh hai trường tại Bồ Đào Nha, không đại diện mặc nhiên cho học sinh Việt Nam. Ngưỡng `G3 < 10` là quy ước vận hành trong nghiên cứu này trên thang điểm 0–20, không phải chuẩn đánh giá phổ quát.

UCI ghi nhận mối liên hệ mạnh giữa G3 và G1/G2. Do nghiên cứu hướng đến kịch bản cảnh báo sớm dựa trên thông tin ngoài điểm số, G1, G2 và G3 không được đưa vào tập đặc trưng. Việc sử dụng G1/G2 sẽ tạo thành một bài toán khác: dự đoán kết quả cuối kỳ sau khi đã có điểm các kỳ trước.

**Trích dẫn:** Cortez, P. (2008). *Student Performance*. UCI Machine Learning Repository. DOI: [10.24432/C5TG7T](https://doi.org/10.24432/C5TG7T).


## 2. Câu hỏi nghiên cứu, nhãn mục tiêu và kiểm soát rò rỉ dữ liệu

### Câu hỏi nghiên cứu

Mức độ hữu ích của các thông tin học tập và hoàn cảnh sẵn có, không bao gồm điểm G1/G2/G3, trong việc nhận diện học sinh có điểm cuối kỳ môn Toán dưới ngưỡng 10/20 là bao nhiêu?

### Định nghĩa nhãn

- `at_risk = 1`: `G3 < 10`, nhóm được xác định là cần quan tâm trong phạm vi nghiên cứu.
- `at_risk = 0`: `G3 >= 10`, nhóm còn lại.

Đây là bài toán **binary classification**. Lớp 1 là lớp trọng tâm về mặt ứng dụng vì bỏ sót học sinh thuộc nhóm này có thể làm giảm giá trị của một hệ thống cảnh báo sớm.

### Kiểm soát data leakage

`G3` chỉ được dùng để tạo nhãn; `G1` và `G2` bị loại khỏi đặc trưng theo thiết kế bài toán. Các phép điền giá trị thiếu, chuẩn hóa và mã hóa được đặt trong `Pipeline` để chỉ học tham số trên phần huấn luyện của từng fold. Tập test được giữ riêng cho đánh giá cuối cùng sau khi đã chốt mô hình và ngưỡng.


## 3. Môi trường thực nghiệm và khả năng tái lập

Quy trình được triển khai bằng Python, pandas/NumPy cho thao tác dữ liệu, Matplotlib/Seaborn cho trực quan hóa và scikit-learn cho tiền xử lý, kiểm tra chéo, mô hình hóa và đánh giá. Các tham số ngẫu nhiên được kiểm soát bằng `RANDOM_STATE`; phiên bản thư viện được khai báo trong `requirements.txt` nhằm tăng khả năng tái lập giữa Google Colab và môi trường cục bộ.

Các lệnh cài đặt chỉ là cấu hình môi trường, không thuộc nội dung suy luận thống kê của nghiên cứu.


In [ ]:
# Nếu cần cài thêm thư viện, bỏ dấu # và chạy lại ô này.
# %pip install -q -r requirements.txt

# Cài thư viện còn thiếu tự động (nếu môi trường Colab đã có đủ thì không cài lại).
import importlib.util
import subprocess
import sys
module_to_package = {
    'pandas': 'pandas', 'numpy': 'numpy', 'matplotlib': 'matplotlib',
    'requests': 'requests', 'bs4': 'beautifulsoup4', 'sklearn': 'scikit-learn',
    'scipy': 'scipy', 'joblib': 'joblib', 'tabulate': 'tabulate'
}
missing_packages = [pkg for module, pkg in module_to_package.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    print('Đang cài thư viện còn thiếu:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])

import os
import json
import html
import hashlib
import warnings
import zipfile
from io import BytesIO
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urljoin, urlparse

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from bs4 import BeautifulSoup
from IPython.display import display, Markdown

from scipy.stats import chi2_contingency
from sklearn.base import clone
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.exceptions import UndefinedMetricWarning
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, fbeta_score, average_precision_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, precision_recall_curve
)
from sklearn.inspection import permutation_importance
import joblib

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)
RANDOM_STATE = 42
PASSING_GRADE = 10
TEST_SIZE = 0.20
N_SPLITS = 5
N_REPEATS = 2
print('Đã nạp thư viện và cấu hình cơ bản.')


## 4. Quản lý đường dẫn và sản phẩm nghiên cứu

Notebook xác định thư mục gốc của repository để đọc dữ liệu và ghi các sản phẩm đầu ra vào các vị trí thống nhất. Cơ chế này cho phép cùng một notebook hoạt động trong thư mục dự án trên Google Colab hoặc trên máy cá nhân mà không cần thay đổi thủ công từng đường dẫn.

Các sản phẩm được tách theo mục đích: dữ liệu nguồn trong `data/`, bảng và biểu đồ trong `outputs/`, báo cáo trong `reports/`, và mô hình đã lưu trong `models/`.


In [ ]:
def tim_thu_muc_goc(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'data').is_dir() and (candidate / 'requirements.txt').exists():
            return candidate
    # Colab thường giữ thư mục dự án con dưới /content, trong khi cwd vẫn là /content.
    content_dir = Path('/content')
    if content_dir.exists():
        for req in content_dir.glob('*/requirements.txt'):
            candidate = req.parent
            if (candidate / 'data').is_dir() and (candidate / 'src').is_dir():
                return candidate
        for req in content_dir.glob('*/*/requirements.txt'):
            candidate = req.parent
            if (candidate / 'data').is_dir() and (candidate / 'src').is_dir():
                return candidate
        return content_dir
    return start

ROOT_DIR = tim_thu_muc_goc()
DATA_DIR = ROOT_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
OUTPUT_DIR = ROOT_DIR / 'outputs'
FIGURES_DIR = OUTPUT_DIR / 'figures'
REPORT_DIR = ROOT_DIR / 'reports'
MODEL_DIR = ROOT_DIR / 'models'
for folder in [DATA_DIR, RAW_DIR, OUTPUT_DIR, FIGURES_DIR, REPORT_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

CSV_PATH = DATA_DIR / 'student-mat.csv'
DEMO_PATH = DATA_DIR / 'student-mat-demo.csv'
UCI_PAGE_URL = 'https://archive.ics.uci.edu/dataset/320/student-performance'
UCI_ARCHIVE_FALLBACK = 'https://archive.ics.uci.edu/static/public/320/student+performance.zip'
DATASET_DOI = '10.24432/C5TG7T'
DATASET_LICENSE = 'CC BY 4.0'
MAX_DOWNLOAD_BYTES = 20 * 1024 * 1024
FORCE_CRAWL = False  # đổi thành True nếu muốn thu thập lại dữ liệu UCI từ đầu
print('Thư mục dự án:', ROOT_DIR)
print('Đường dẫn CSV chính:', CSV_PATH)


## 5. Thu thập dữ liệu và truy xuất nguồn

Quy trình thu thập lấy trang dữ liệu công khai của UCI bằng `requests`, phân tích cấu trúc HTML bằng `BeautifulSoup`, rồi tải và giải nén tệp ZIP để lấy `student-mat.csv`. Mã chỉ truy cập trang dữ liệu và tệp công khai, không đăng nhập và không thu thập hồ sơ cá nhân trực tiếp từ người dùng website.

HTML, ZIP, CSV gốc và siêu dữ liệu về nguồn được lưu lại để hỗ trợ truy xuất nguồn gốc (data provenance), xác minh phiên bản và kiểm tra tính toàn vẹn của dữ liệu.


In [ ]:
def tim_csv_trong_zip(du_lieu_zip: bytes, ten_can_tim='student-mat.csv', do_sau=0) -> bytes:
    """Tìm CSV trong ZIP, kể cả khi có ZIP lồng bên trong."""
    if do_sau > 4:
        raise ValueError('ZIP lồng quá nhiều lớp; dừng để an toàn.')
    with zipfile.ZipFile(BytesIO(du_lieu_zip)) as zf:
        ten_tep = zf.namelist()
        match = next((name for name in ten_tep if Path(name).name.lower() == ten_can_tim.lower()), None)
        if match:
            return zf.read(match)
        for child in [name for name in ten_tep if name.lower().endswith('.zip')]:
            try:
                return tim_csv_trong_zip(zf.read(child), ten_can_tim, do_sau + 1)
            except (zipfile.BadZipFile, FileNotFoundError, ValueError):
                continue
    raise FileNotFoundError(f'Không tìm thấy {ten_can_tim} trong ZIP.')


def thu_thap_du_lieu_uci():
    """Crawl trang UCI và tải bộ dữ liệu công khai; ghi nhận nguồn và mã SHA-256."""
    session = requests.Session()
    session.headers.update({
        'User-Agent': 'StudentLearningMLStudentProject/1.0 (educational, low-volume retrieval)',
        'Accept': 'text/html,application/zip,*/*'
    })
    html_trang = ''
    url_zip = None
    tim_thay_link_tren_trang = False
    try:
        response = session.get(UCI_PAGE_URL, timeout=25)
        response.raise_for_status()
        html_trang = response.text
        (RAW_DIR / 'trang_gioi_thieu_uci.html').write_text(html_trang, encoding='utf-8')
        soup = BeautifulSoup(html_trang, 'html.parser')
        ung_vien = []
        for a in soup.find_all('a', href=True):
            href = html.unescape(a['href'].strip())
            url = urljoin(UCI_PAGE_URL, href)
            path = urlparse(url).path.lower()
            if path.endswith('.zip') or ('/static/public/320/' in path and 'student' in path):
                ung_vien.append(url)
        url_zip = next((u for u in ung_vien if '320' in u and 'student' in u.lower()), None)
        tim_thay_link_tren_trang = url_zip is not None
    except requests.RequestException as exc:
        print('Chưa đọc được trang giới thiệu; sẽ thử địa chỉ tải UCI dự phòng. Chi tiết:', exc)

    if not url_zip:
        url_zip = UCI_ARCHIVE_FALLBACK
    resp_zip = session.get(url_zip, timeout=25)
    resp_zip.raise_for_status()
    if len(resp_zip.content) > MAX_DOWNLOAD_BYTES:
        raise ValueError('Tệp tải về vượt quá giới hạn 20 MB của bài tập.')
    (RAW_DIR / 'student-performance.zip').write_bytes(resp_zip.content)
    csv_bytes = tim_csv_trong_zip(resp_zip.content)
    (RAW_DIR / 'student-mat-goc.csv').write_bytes(csv_bytes)
    df_tai = pd.read_csv(BytesIO(csv_bytes), sep=';')
    if 'G3' not in df_tai.columns:
        raise ValueError('CSV tải về không có cột G3; dừng để kiểm tra nguồn dữ liệu.')
    df_tai.to_csv(CSV_PATH, sep=';', index=False, encoding='utf-8')
    metadata = {
        'ten_bo_du_lieu': 'Student Performance - Mathematics',
        'trang_nguon': UCI_PAGE_URL,
        'doi': DATASET_DOI,
        'giay_phep': DATASET_LICENSE,
        'url_zip': url_zip,
        'cach_tim_duong_dan_tai': 'tim_thay_tren_trang_gioi_thieu' if tim_thay_link_tren_trang else 'dung_duong_dan_uci_du_phong',
        'thoi_gian_thu_thap_utc': datetime.now(timezone.utc).isoformat(),
        'so_dong': int(df_tai.shape[0]),
        'so_cot': int(df_tai.shape[1]),
        'sha256_csv_su_dung': hashlib.sha256(CSV_PATH.read_bytes()).hexdigest(),
        'ghi_chu': 'Bộ dữ liệu gốc từ hai trường trung học ở Bồ Đào Nha; không đại diện trực tiếp cho học sinh Việt Nam.'
    }
    (RAW_DIR / 'thong_tin_thu_thap.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
    source_mode = 'official_uci_crawl' if tim_thay_link_tren_trang else 'official_uci_direct_download'
    return df_tai, source_mode

print('Đã khai báo hàm crawl. Ô tiếp theo sẽ tải UCI nếu chưa có CSV chính thức.')


## 6. Nạp dữ liệu và xác minh nguồn

Khi kết nối Internet khả dụng, notebook ưu tiên thu thập CSV từ nguồn UCI. Nếu việc thu thập thất bại, dữ liệu minh họa đi kèm repository có thể được dùng để kiểm tra khả năng thực thi của mã. Chương trình phân biệt rõ dữ liệu chính thức với dữ liệu minh họa; kết quả từ dữ liệu minh họa không được diễn giải như kết quả benchmark của bộ dữ liệu UCI.

Trạng thái nguồn dữ liệu được lưu kèm kết quả để tránh nhầm lẫn giữa kiểm thử phần mềm và thực nghiệm chính thức.


In [ ]:
nguon_du_lieu = None
metadata_path = RAW_DIR / 'thong_tin_thu_thap.json'

if CSV_PATH.exists() and not FORCE_CRAWL:
    df = pd.read_csv(CSV_PATH, sep=';')
    nguon_du_lieu = 'local_csv_not_verified'
    if metadata_path.exists():
        try:
            metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
            sha = hashlib.sha256(CSV_PATH.read_bytes()).hexdigest()
            if 'archive.ics.uci.edu' in metadata.get('trang_nguon', '') and metadata.get('doi') == DATASET_DOI and metadata.get('sha256_csv_su_dung') == sha:
                nguon_du_lieu = 'local_official_csv'
        except (OSError, json.JSONDecodeError):
            pass
else:
    try:
        df, nguon_du_lieu = thu_thap_du_lieu_uci()
        print('Đã thu thập CSV từ nguồn UCI.')
    except Exception as exc:
        print('Không tải được CSV chính thức từ UCI. Chi tiết:', repr(exc))
        if CSV_PATH.exists():
            df = pd.read_csv(CSV_PATH, sep=';')
            nguon_du_lieu = 'local_csv_not_verified'
        elif DEMO_PATH.exists():
            df = pd.read_csv(DEMO_PATH, sep=';')
            nguon_du_lieu = 'synthetic_demo_not_for_final_benchmark'
            print('\nCẢNH BÁO: Đang dùng CSV minh họa. Chỉ dùng để kiểm tra code, không dùng số liệu này làm kết quả thi chính thức.')
        else:
            raise RuntimeError('Không có CSV để chạy. Hãy kiểm tra Internet hoặc tải student-mat.csv vào thư mục data/.')

if 'synthetic_demo' in nguon_du_lieu:
    print('Nguồn dữ liệu: DEMO — không phải benchmark chính thức.')
elif 'official' in nguon_du_lieu:
    print('Nguồn dữ liệu: UCI chính thức hoặc CSV đã xác minh metadata.')
else:
    print('Nguồn dữ liệu chưa xác minh bằng metadata:', nguon_du_lieu)
print('Kích thước:', df.shape)
display(df.head())


## 7. Hồ sơ dữ liệu và chất lượng dữ liệu

Phân tích hồ sơ dữ liệu xác định số lượng mẫu/cột, dòng trùng hoàn toàn, giá trị thiếu và các nhóm biến quan sát. Đây là bước mô tả dữ liệu trước mô hình hóa, giúp đánh giá độ đầy đủ và hiểu đúng đơn vị quan sát. Một dòng trùng không tự động đồng nghĩa với bản ghi sai; quyết định loại bỏ cần dựa trên ý nghĩa của từng trường và quy trình thu thập.


In [ ]:
print('Số học sinh:', len(df))
print('Số cột nguồn:', len(df.columns))
print('Số dòng trùng hoàn toàn:', int(df.duplicated().sum()))
print('Tổng số giá trị thiếu:', int(df.isna().sum().sum()))

mo_ta_bo_du_lieu = pd.DataFrame([
    ['Nguồn', 'UCI Machine Learning Repository — Student Performance'],
    ['Tập đang dùng', 'student-mat.csv — môn Toán'],
    ['Quy mô gốc được công bố', '395 học sinh, 33 cột cho tập môn Toán'],
    ['Đầu vào dự kiến', '30 biến không phải điểm số'],
    ['Không đưa vào đầu vào', 'G1, G2, G3'],
    ['Nhãn', 'at_risk = 1 nếu G3 < 10; ngược lại là 0'],
    ['Phạm vi gốc', 'Hai trường trung học ở Bồ Đào Nha'],
    ['Giấy phép', 'CC BY 4.0 — cần ghi nguồn'],
], columns=['Mục', 'Mô tả'])
display(mo_ta_bo_du_lieu)
print('\nKiểu dữ liệu và số giá trị thiếu theo cột:')
display(pd.DataFrame({'kiểu_dữ_liệu': df.dtypes.astype(str), 'số_giá_trị_thiếu': df.isna().sum()}))

print('\nThống kê nhanh các cột số:')
display(df.select_dtypes(include=np.number).describe().T.round(2))


## 8. Xây dựng nhãn mục tiêu và phân bố lớp

Điểm cuối kỳ `G3` nằm trên thang điểm 0–20. Nhãn nhị phân được xây dựng theo ngưỡng định trước: `G3 < 10` được mã hóa thành lớp 1 và `G3 >= 10` thành lớp 0. Tỷ lệ của hai lớp được kiểm tra trước khi huấn luyện vì mức độ mất cân bằng ảnh hưởng đến cách diễn giải Accuracy, Precision và Recall.


In [ ]:
PASSING_GRADE = 10
data = df.copy()
data['G3'] = pd.to_numeric(data['G3'], errors='coerce')
data = data.dropna(subset=['G3']).copy()
data['at_risk'] = (data['G3'] < PASSING_GRADE).astype(int)

bang_nhan = data['at_risk'].value_counts().reindex([0, 1], fill_value=0).rename(index={
    0: 'Lớp 0 — G3 >= 10',
    1: 'Lớp 1 — G3 < 10'
}).to_frame('Số học sinh')
bang_nhan['Tỷ lệ'] = bang_nhan['Số học sinh'] / len(data)
display(bang_nhan)
print('Nếu lớp 0 đông hơn lớp 1, mô hình có thể đạt accuracy khá cao nhưng vẫn bỏ sót nhiều học sinh lớp 1. Vì vậy phải đọc Recall/F1 theo từng lớp.')

fig, ax = plt.subplots(figsize=(7, 4))
counts = data['at_risk'].value_counts().reindex([0, 1], fill_value=0)
ax.bar(['Lớp 0: G3 >= 10', 'Lớp 1: G3 < 10'], counts.values)
ax.set_ylabel('Số học sinh')
ax.set_title('Phân bố nhãn cần dự đoán')
for i, v in enumerate(counts.values):
    ax.text(i, v + max(counts.values) * .015, f'{v} ({v/len(data):.1%})', ha='center')
plt.tight_layout(); plt.show()


## 9. Exploratory Data Analysis (EDA)

EDA mô tả phân bố của nhãn và đặc trưng, đồng thời khảo sát sự khác biệt giữa các nhóm học sinh theo thời gian tự học, số lần trượt, số buổi nghỉ và các yếu tố hỗ trợ. Các biểu đồ được dùng để xác định mẫu hình cần kiểm tra trong phân tích tiếp theo; chúng phản ánh mối liên hệ quan sát được, không chứng minh quan hệ nhân quả.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
rate_by_study = data.groupby('studytime', observed=False)['at_risk'].mean().sort_index()
ax.bar(rate_by_study.index.astype(str), rate_by_study.values)
ax.set_title('Tỷ lệ G3 < 10 theo mức tự học')
ax.set_xlabel('Mức tự học/tuần (1 thấp nhất, 4 cao nhất)')
ax.set_ylabel('Tỷ lệ lớp 1')
ax.set_ylim(0, 1); plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
data.boxplot(column='absences', by='at_risk', ax=ax)
ax.set_title('Số buổi nghỉ theo nhãn')
ax.set_xlabel('Nhãn (0: G3 >= 10; 1: G3 < 10)')
ax.set_ylabel('Số buổi nghỉ')
plt.suptitle(''); plt.tight_layout(); plt.show()

print('Hãy mô tả điều nhìn thấy trên biểu đồ nhưng tránh khẳng định quan hệ nhân quả. Ví dụ: “Trong mẫu này, tỷ lệ lớp 1 khác nhau giữa các nhóm tự học” — không phải “tự học chắc chắn làm tăng điểm”.')


## 10. Thiết kế đặc trưng và phân chia dữ liệu

Hai tập đặc trưng được xác định trước khi so sánh bằng kiểm tra chéo:

1. **Nhóm học tập cốt lõi — 6 biến:** `studytime`, `failures`, `absences`, `schoolsup`, `famsup`, `paid`. Nhóm này gọn, dễ diễn giải và gần với những yếu tố có thể dùng trong hoạt động hỗ trợ học tập.
2. **Nhóm mở rộng — 30 biến không phải điểm:** gồm thông tin cá nhân, gia đình, trường học, thói quen và hỗ trợ. Nhóm mở rộng có thể chứa tín hiệu bổ sung nhưng cũng làm tăng độ phức tạp, nhiễu và các câu hỏi về công bằng.

Liên hệ đơn biến không được dùng làm tiêu chí duy nhất để giữ hoặc loại đặc trưng. Hai tập được đánh giá bằng cùng thiết kế kiểm tra chéo. Hai mươi phần trăm dữ liệu được giữ làm test và không tham gia chọn đặc trưng, mô hình hoặc ngưỡng.


In [ ]:
NON_GRADE_FEATURES = [
    'school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu',
    'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures',
    'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet',
    'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences'
]
CORE_FEATURES = ['studytime', 'failures', 'absences', 'schoolsup', 'famsup', 'paid']
NUMERIC_FEATURES = [
    'age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel',
    'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences'
]
CATEGORICAL_FEATURES = [
    'school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason',
    'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher',
    'internet', 'romantic'
]

missing_features = [c for c in NON_GRADE_FEATURES + ['G3'] if c not in data.columns]
if missing_features:
    raise ValueError(f'Dữ liệu thiếu cột bắt buộc: {missing_features}')
X = data[NON_GRADE_FEATURES].copy()
y = data['at_risk'].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
print(f'Tập huấn luyện: {len(X_train)} dòng; tập kiểm tra cuối cùng: {len(X_test)} dòng.')
print('Tỷ lệ lớp 1 trong train:', f'{y_train.mean():.1%}')
print('Tỷ lệ lớp 1 trong test :', f'{y_test.mean():.1%}')
print('Không đọc kết quả test để lựa chọn mô hình hay đặc trưng.')

feature_sets = {
    'Nhóm học tập cốt lõi (6 biến)': CORE_FEATURES,
    'Toàn bộ biến không phải điểm (30 biến)': NON_GRADE_FEATURES,
}
feature_size = pd.DataFrame([{'Nhóm đặc trưng': name, 'Số biến': len(cols), 'Các biến': ', '.join(cols)} for name, cols in feature_sets.items()])
display(feature_size)


## 11. Phân tích mối liên hệ giữa đặc trưng và nhãn trên tập train

Phân tích này mô tả mối liên hệ đơn biến giữa từng đặc trưng và nhãn mục tiêu, chỉ sử dụng phần train để tránh đưa thông tin từ tập test vào quá trình khám phá.

- Với biến số/thứ bậc, tương quan Pearson với nhãn nhị phân tương đương hệ số point-biserial trong trường hợp này. Hệ số dương biểu thị giá trị cao đi cùng lớp 1 nhiều hơn trong mẫu; hệ số âm biểu thị xu hướng ngược lại.
- Với biến phân loại, Cramér's V đo mức độ liên hệ giữa các nhóm của biến và nhãn, nhưng không thể hiện hướng dương/âm.

Hai thước đo có cách diễn giải khác nhau và không nên xếp hạng trực tiếp trong cùng một bảng. Tương quan không cho biết quan hệ nhân quả, cũng không đảm bảo đặc trưng sẽ cải thiện hiệu quả dự đoán ngoài mẫu; hiệu quả dự đoán được đánh giá riêng bằng kiểm tra chéo.


In [ ]:
def tinh_cramers_v(x, y):
    bang = pd.crosstab(x.fillna('(thiếu)'), y)
    if bang.empty or min(bang.shape) < 2:
        return 0.0
    chi2 = chi2_contingency(bang, correction=False)[0]
    n = bang.to_numpy().sum()
    mau_so = n * min(bang.shape[0] - 1, bang.shape[1] - 1)
    return float(np.sqrt(chi2 / mau_so)) if mau_so > 0 else 0.0

bien_so_train = [c for c in NUMERIC_FEATURES if c in X_train.columns]
bang_tuong_quan = pd.DataFrame([
    {'Đặc trưng': c, 'Tương quan Pearson với nhãn 1': X_train[c].corr(y_train)}
    for c in bien_so_train
])
bang_tuong_quan['Độ lớn tuyệt đối'] = bang_tuong_quan['Tương quan Pearson với nhãn 1'].abs()
bang_tuong_quan = bang_tuong_quan.sort_values('Độ lớn tuyệt đối', ascending=False)
display(bang_tuong_quan.round(3))
bang_tuong_quan.to_csv(OUTPUT_DIR / 'lien_he_bien_so_voi_nhan_train.csv', index=False)

fig, ax = plt.subplots(figsize=(8, 5))
plot_numeric = bang_tuong_quan.head(10).sort_values('Tương quan Pearson với nhãn 1')
ax.barh(plot_numeric['Đặc trưng'], plot_numeric['Tương quan Pearson với nhãn 1'])
ax.axvline(0, color='black', linewidth=.8)
ax.set_xlabel('Tương quan với nhãn 1 (G3 < 10)')
ax.set_title('Mối liên hệ của biến số/thứ bậc với nhãn — train')
plt.tight_layout(); plt.show()

bang_categorical = pd.DataFrame([
    {'Đặc trưng': c, 'Cramers_V': tinh_cramers_v(X_train[c], y_train)}
    for c in CATEGORICAL_FEATURES if c in X_train.columns
]).sort_values('Cramers_V', ascending=False)
display(bang_categorical.round(3))
bang_categorical.to_csv(OUTPUT_DIR / 'lien_he_bien_phan_loai_voi_nhan_train.csv', index=False)

fig, ax = plt.subplots(figsize=(8, 5))
plot_cat = bang_categorical.head(10).sort_values('Cramers_V')
ax.barh(plot_cat['Đặc trưng'], plot_cat['Cramers_V'])
ax.set_xlim(0, 1)
ax.set_xlabel("Cramér's V (càng gần 1, mức liên hệ trong mẫu càng mạnh)")
ax.set_title('Mối liên hệ của biến phân loại với nhãn — train')
plt.tight_layout(); plt.show()

print('Lưu ý: biến có mức liên hệ cao chưa chắc giúp dự đoán tốt hơn ngoài mẫu. Vì vậy, ô tiếp theo kiểm tra trực tiếp bằng kiểm tra chéo.')


## 12. Tiền xử lý và các mô hình được khảo sát

Các bước xử lý được đóng gói trong `Pipeline`, giúp việc điền giá trị thiếu, chuẩn hóa và mã hóa được khớp riêng trong từng fold thay vì sử dụng thông tin từ toàn bộ dữ liệu.

| Mô hình | Vai trò trong thiết kế thực nghiệm | Hạn chế cần theo dõi |
|---|---|---|
| **Dummy Classifier** | Baseline dựa trên lớp phổ biến nhất, tạo mốc hiệu quả tối thiểu | Không học quan hệ giữa đặc trưng và nhãn |
| **Logistic Regression** | Mô hình phân loại tuyến tính, xác suất đầu ra có thể diễn giải | Ranh giới quyết định cơ bản tuyến tính trên log-odds; có thể bỏ sót quan hệ phi tuyến |
| **Decision Tree** | Học quy tắc chia nhánh và thể hiện tương tác phi tuyến | Cây sâu có thể overfit và nhạy với thay đổi dữ liệu |
| **Random Forest** | Tổng hợp nhiều cây được huấn luyện có tính ngẫu nhiên, nhằm giảm phương sai so với cây đơn | Khó diễn giải trực tiếp hơn Decision Tree; không bảo đảm tốt hơn khi dữ liệu nhỏ |

Các mô hình đại diện cho những giả định và độ phức tạp khác nhau. Chúng được so sánh trên cùng các fold và cùng các tập đặc trưng, thay vì giả định trước mô hình nào sẽ tốt nhất.


In [ ]:
def tao_bo_tien_xu_ly(cac_cot):
    cot_so = [c for c in cac_cot if c in NUMERIC_FEATURES]
    cot_phan_loai = [c for c in cac_cot if c in CATEGORICAL_FEATURES]
    cac_buoc = []
    if cot_so:
        quy_trinh_so = Pipeline([
            ('dien_thieu', SimpleImputer(strategy='median')),
            ('chuan_hoa', StandardScaler())
        ])
        cac_buoc.append(('bien_so', quy_trinh_so, cot_so))
    if cot_phan_loai:
        quy_trinh_phan_loai = Pipeline([
            ('dien_thieu', SimpleImputer(strategy='most_frequent')),
            ('ma_hoa', OneHotEncoder(handle_unknown='ignore'))
        ])
        cac_buoc.append(('bien_phan_loai', quy_trinh_phan_loai, cot_phan_loai))
    return ColumnTransformer(cac_buoc, remainder='drop')


def tao_pipeline(mo_hinh, cac_cot):
    return Pipeline([
        ('tien_xu_ly', tao_bo_tien_xu_ly(cac_cot)),
        ('mo_hinh', clone(mo_hinh))
    ])

models = {
    'Dummy Classifier': DummyClassifier(strategy='most_frequent'),
    'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE),
    'Decision Tree': DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=5, min_samples_leaf=4, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=1),
}

scoring = {
    'accuracy': 'accuracy',
    'balanced_accuracy': 'balanced_accuracy',
    'precision_at_risk': 'precision',
    'recall_at_risk': 'recall',
    'f1_at_risk': 'f1',
    'average_precision': 'average_precision',
}
cv_lap_lai = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)
print(f'Sẵn sàng so sánh bằng {N_SPLITS}-fold CV lặp {N_REPEATS} lần trên tập train.')


## 13. So sánh mô hình và tập đặc trưng bằng kiểm tra chéo

Một lần chia ngẫu nhiên có thể tạo ra kết quả biến động lớn khi dữ liệu chỉ có vài trăm mẫu. Vì vậy, phần train được đánh giá bằng `RepeatedStratifiedKFold` với 5 folds và 2 lần lặp. Phân tầng giúp giữ tỷ lệ lớp tương đối tương tự giữa các fold.

- Kiểm tra chéo chỉ được thực hiện trên phần train; test vẫn độc lập.
- **Average Precision** đánh giá khả năng xếp hạng mẫu lớp 1 trên nhiều ngưỡng.
- **F1 và Recall lớp 1** trong bảng phản ánh hiệu quả tại ngưỡng mặc định 0,5.
- Trung bình và độ lệch chuẩn qua các fold cùng được báo cáo để mô tả hiệu quả điển hình và mức độ biến động.


In [ ]:
dong_ket_qua = []
with warnings.catch_warnings():
    warnings.filterwarnings('ignore', category=UndefinedMetricWarning)
    for ten_mo_hinh, estimator in models.items():
        if ten_mo_hinh == 'Dummy Classifier':
            cac_nhom = {'Dummy Classifier': NON_GRADE_FEATURES}
        else:
            cac_nhom = feature_sets
        for ten_nhom, cac_cot in cac_nhom.items():
            pipeline = tao_pipeline(estimator, cac_cot)
            ket_qua_cv = cross_validate(
                pipeline, X_train[cac_cot], y_train,
                scoring=scoring, cv=cv_lap_lai, n_jobs=1, return_train_score=False
            )
            dong = {
                'Mô hình': ten_mo_hinh,
                'Nhóm đặc trưng': ten_nhom,
                'Số biến': len(cac_cot),
            }
            for metric_name in scoring:
                values = ket_qua_cv[f'test_{metric_name}']
                dong[f'{metric_name}_TB'] = float(np.mean(values))
                dong[f'{metric_name}_ĐLC'] = float(np.std(values, ddof=1))
            dong_ket_qua.append(dong)

bang_cv = pd.DataFrame(dong_ket_qua)
bang_cv = bang_cv.sort_values(['average_precision_TB', 'f1_at_risk_TB', 'Số biến'], ascending=[False, False, True]).reset_index(drop=True)
bang_cv.to_csv(OUTPUT_DIR / 'so_sanh_kiem_tra_cheo.csv', index=False)
display(bang_cv.round(3))


## 14. Tiêu chí lựa chọn mô hình

Quy tắc lựa chọn được xác định trước khi xem kết quả test:

1. Xếp hạng ứng viên theo Average Precision trung bình qua các folds.
2. Khi kết quả gần nhau, xét F1 của lớp 1.
3. Nếu vẫn tương đương, ưu tiên tập đặc trưng gọn hơn vì khả năng giải thích và tính tiết kiệm dữ liệu.

Đây là một lựa chọn phù hợp với mục tiêu xếp hạng lớp cần quan tâm, không phải quy tắc tối ưu cho mọi ứng dụng. Trong bối cảnh mà chi phí bỏ sót lớn hơn chi phí cảnh báo nhầm, Recall có thể được đặt làm tiêu chí chính; khi đó lý do lựa chọn và số cảnh báo nhầm cần được báo cáo rõ ràng.


In [ ]:
ung_vien = bang_cv[bang_cv['Mô hình'] != 'Dummy Classifier'].copy()
ung_vien = ung_vien.sort_values(['average_precision_TB', 'f1_at_risk_TB', 'Số biến'], ascending=[False, False, True]).reset_index(drop=True)
phuong_an = ung_vien.iloc[0]
ten_mo_hinh_chon = str(phuong_an['Mô hình'])
ten_nhom_chon = str(phuong_an['Nhóm đặc trưng'])
cot_chon = feature_sets[ten_nhom_chon]
mo_hinh_goc = models[ten_mo_hinh_chon]
pipeline_chon = tao_pipeline(mo_hinh_goc, cot_chon)

print('Phương án được chọn từ CV, chưa xem test:')
print('Mô hình:', ten_mo_hinh_chon)
print('Nhóm đặc trưng:', ten_nhom_chon)
print('Số đặc trưng:', len(cot_chon))
print(f"Average Precision CV: {phuong_an['average_precision_TB']:.3f} ± {phuong_an['average_precision_ĐLC']:.3f}")
print(f"F1 lớp 1 CV: {phuong_an['f1_at_risk_TB']:.3f} ± {phuong_an['f1_at_risk_ĐLC']:.3f}")
print(f"Recall lớp 1 CV: {phuong_an['recall_at_risk_TB']:.3f} ± {phuong_an['recall_at_risk_ĐLC']:.3f}")

# Biểu đồ trung bình Average Precision, thanh lỗi là độ lệch chuẩn qua các fold.
plot_cv = bang_cv.copy()
plot_cv['Nhãn biểu đồ'] = plot_cv['Mô hình'] + '\n' + plot_cv['Nhóm đặc trưng']
plot_cv = plot_cv.sort_values('average_precision_TB')
fig, ax = plt.subplots(figsize=(10, max(5, .55 * len(plot_cv))))
ax.barh(plot_cv['Nhãn biểu đồ'], plot_cv['average_precision_TB'], xerr=plot_cv['average_precision_ĐLC'], capsize=3)
ax.set_xlim(0, 1)
ax.set_xlabel('Average Precision trung bình ± độ lệch chuẩn')
ax.set_title('So sánh mô hình/nhóm đặc trưng qua kiểm tra chéo')
plt.tight_layout(); plt.savefig(FIGURES_DIR / '06_so_sanh_kiem_tra_cheo.png', dpi=160); plt.show()


## 15. Lựa chọn ngưỡng dự đoán từ dữ liệu huấn luyện

Ngưỡng 0,5 là quy ước phổ biến, không phải lựa chọn tối ưu mặc định cho mọi bài toán. Khi thay đổi ngưỡng, số dự đoán thuộc lớp 1 thay đổi, tạo ra đánh đổi giữa Precision và Recall.

Để không lựa chọn ngưỡng dựa trên test, notebook tạo dự đoán ngoài mẫu (**out-of-fold, OOF**) cho các quan sát train bằng 5-fold CV. Ngưỡng được lựa chọn theo F1 lớp 1, với Recall là tiêu chí phụ khi F1 tương đương. Ngưỡng thấp hơn thường có thể tăng Recall nhưng đồng thời làm tăng cảnh báo nhầm; vì vậy phải diễn giải các chỉ số cùng nhau.


In [ ]:
cv_oof = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE + 1)
with warnings.catch_warnings():
    warnings.filterwarnings('ignore', category=UndefinedMetricWarning)
    xac_suat_oof = cross_val_predict(
        pipeline_chon, X_train[cot_chon], y_train,
        cv=cv_oof, method='predict_proba', n_jobs=1
    )[:, 1]

def tinh_chi_so(y_true, xac_suat, nguong=0.5):
    du_doan = (np.asarray(xac_suat) >= nguong).astype(int)
    return {
        'Accuracy': accuracy_score(y_true, du_doan),
        'Balanced accuracy': balanced_accuracy_score(y_true, du_doan),
        'Precision lớp 1': precision_score(y_true, du_doan, zero_division=0),
        'Recall lớp 1': recall_score(y_true, du_doan, zero_division=0),
        'F1 lớp 1': f1_score(y_true, du_doan, zero_division=0),
        'F2 lớp 1': fbeta_score(y_true, du_doan, beta=2, zero_division=0),
        'Average Precision': average_precision_score(y_true, xac_suat),
    }

cac_nguong = []
for nguong in np.arange(0.15, 0.851, 0.025):
    dong = {'Ngưỡng': float(nguong), **tinh_chi_so(y_train, xac_suat_oof, float(nguong))}
    cac_nguong.append(dong)
bang_nguong = pd.DataFrame(cac_nguong).sort_values(
    ['F1 lớp 1', 'Recall lớp 1', 'Precision lớp 1', 'Ngưỡng'],
    ascending=[False, False, False, False]
).reset_index(drop=True)
nguong_chon = float(bang_nguong.iloc[0]['Ngưỡng'])
bang_nguong.to_csv(OUTPUT_DIR / 'so_sanh_nguong_train_oof.csv', index=False)
print('Một số ngưỡng hàng đầu trên dự đoán OOF của train:')
display(bang_nguong.head(10).round(3))
print('Ngưỡng được chọn theo F1, Recall phụ:', nguong_chon)


## 16. Đánh giá cuối cùng trên tập test độc lập

Sau khi chốt tập đặc trưng, thuật toán và ngưỡng, mô hình được khớp trên toàn bộ phần train rồi đánh giá trên test một lần. Từ thời điểm này, kết quả test được xem là ước lượng cuối cùng trong thiết kế hiện tại; các điều chỉnh tiếp theo cần được xác nhận bằng một quy trình đánh giá mới, không chọn lại dựa trên chính test này.

Báo cáo gồm Accuracy, Balanced Accuracy, Precision, Recall, F1 và Average Precision. Các chỉ số được đọc cùng ma trận nhầm lẫn để xác định loại sai sót mà mô hình tạo ra.


In [ ]:
pipeline_chon.fit(X_train[cot_chon], y_train)
xac_suat_test = pipeline_chon.predict_proba(X_test[cot_chon])[:, 1]
du_doan_test = (xac_suat_test >= nguong_chon).astype(int)
chi_so_test = tinh_chi_so(y_test, xac_suat_test, nguong_chon)

# So sánh với baseline luôn dự đoán lớp phổ biến nhất.
baseline_test = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
du_doan_baseline = baseline_test.predict(X_test)
bang_doi_chieu_test = pd.DataFrame([
    {'Mô hình': 'Baseline — luôn chọn lớp phổ biến',
     'Accuracy': accuracy_score(y_test, du_doan_baseline),
     'Balanced accuracy': balanced_accuracy_score(y_test, du_doan_baseline),
     'Precision lớp 1': precision_score(y_test, du_doan_baseline, zero_division=0),
     'Recall lớp 1': recall_score(y_test, du_doan_baseline, zero_division=0),
     'F1 lớp 1': f1_score(y_test, du_doan_baseline, zero_division=0)},
    {'Mô hình': f'{ten_mo_hinh_chon} — đã chọn qua CV',
     'Accuracy': chi_so_test['Accuracy'],
     'Balanced accuracy': chi_so_test['Balanced accuracy'],
     'Precision lớp 1': chi_so_test['Precision lớp 1'],
     'Recall lớp 1': chi_so_test['Recall lớp 1'],
     'F1 lớp 1': chi_so_test['F1 lớp 1']},
])
display(bang_doi_chieu_test.round(3))
bang_doi_chieu_test.to_csv(OUTPUT_DIR / 'so_sanh_baseline_va_mo_hinh_test.csv', index=False)

print('Mô hình:', ten_mo_hinh_chon)
print('Nhóm đặc trưng:', ten_nhom_chon)
print('Ngưỡng đã chốt từ train OOF:', round(nguong_chon, 3))
print('\nChỉ số trên test:')
display(pd.DataFrame([chi_so_test]).T.rename(columns={0: 'Giá trị'}).round(3))


## 17. Phân tích kết quả theo từng lớp

Lớp 0 (`G3 >= 10`) và lớp 1 (`G3 < 10`) thể hiện hai nhóm có ý nghĩa khác nhau đối với câu hỏi nghiên cứu. Do mục tiêu là cảnh báo sớm, chất lượng dự đoán lớp 1 cần được phân tích riêng thay vì suy ra từ Accuracy tổng thể.

- **Recall lớp 1 thấp:** mô hình bỏ sót nhiều học sinh thực sự thuộc nhóm cần quan tâm (False Negatives, FN).
- **Precision lớp 1 thấp:** trong số học sinh được cảnh báo, tỷ lệ cảnh báo nhầm cao (False Positives, FP).
- **Chỉ số lớp 0 cao nhưng lớp 1 thấp:** mô hình nhận diện nhóm còn lại tốt hơn nhóm cần quan tâm; đây là bất cân xứng hiệu quả cần được nêu rõ trong phần thảo luận.

Diễn giải cuối cùng cần dựa trên các chỉ số và số lượng FN/FP thực tế của lần chạy dữ liệu chính thức. Không thể kết luận mô hình có giá trị ứng dụng chỉ từ Accuracy cao.


In [ ]:
bao_cao_lop_dict = classification_report(
    y_test, du_doan_test, labels=[0, 1], zero_division=0, output_dict=True
)
bang_chi_so_lop = pd.DataFrame(bao_cao_lop_dict).T.rename(index={
    '0': 'Lớp 0 — G3 >= 10',
    '1': 'Lớp 1 — G3 < 10',
})
display(bang_chi_so_lop.round(3))
bang_chi_so_lop.to_csv(OUTPUT_DIR / 'chi_so_theo_tung_lop.csv')
print(classification_report(
    y_test, du_doan_test, labels=[0, 1],
    target_names=['Lớp 0 — G3 >= 10', 'Lớp 1 — G3 < 10'], zero_division=0
))

cm = confusion_matrix(y_test, du_doan_test, labels=[0, 1])
tn, fp, fn, tp = [int(v) for v in cm.ravel()]
print(f'TN={tn}: dự đoán lớp 0 và đúng.')
print(f'FP={fp}: dự đoán lớp 1 nhưng thực tế là lớp 0 — cảnh báo nhầm.')
print(f'FN={fn}: thực tế lớp 1 nhưng mô hình dự đoán lớp 0 — bỏ sót nhóm cần quan tâm.')
print(f'TP={tp}: dự đoán lớp 1 và đúng.')

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ConfusionMatrixDisplay(cm, display_labels=['Lớp 0: G3 >= 10', 'Lớp 1: G3 < 10']).plot(ax=ax, colorbar=False, values_format='d')
ax.set_title(f'Ma trận nhầm lẫn — {ten_mo_hinh_chon}')
ax.set_xlabel('Nhãn dự đoán'); ax.set_ylabel('Nhãn thực tế')
plt.tight_layout(); plt.savefig(FIGURES_DIR / '07_ma_tran_nham_lan_test.png', dpi=160); plt.show()

precision0 = float(bao_cao_lop_dict['0']['precision'])
recall0 = float(bao_cao_lop_dict['0']['recall'])
f1_0 = float(bao_cao_lop_dict['0']['f1-score'])
precision1 = float(bao_cao_lop_dict['1']['precision'])
recall1 = float(bao_cao_lop_dict['1']['recall'])
f1_1 = float(bao_cao_lop_dict['1']['f1-score'])
print('\nINSIGHT TỰ ĐỘNG TỪ KẾT QUẢ:')
print(f'- F1 lớp 0 = {f1_0:.1%}; F1 lớp 1 = {f1_1:.1%}.')
if f1_1 < f1_0 - 0.10:
    print(f'- F1 lớp 1 thấp hơn rõ rệt F1 lớp 0. Precision/Recall lớp 1 là {precision1:.1%}/{recall1:.1%}; xem chỉ số nào là nguyên nhân chính khiến lớp 1 kém hơn.')
elif f1_0 < f1_1 - 0.10:
    print(f'- F1 lớp 0 thấp hơn rõ rệt F1 lớp 1; cần xem mô hình có cảnh báo quá rộng hay không.')
else:
    print('- F1 hai lớp tương đối gần nhau.')
if recall1 < recall0 - 0.10:
    print(f'- Recall lớp 1 ({recall1:.1%}) thấp hơn recall lớp 0 ({recall0:.1%}). Mô hình bỏ sót {fn} học sinh thực sự có nguy cơ. Đây là hạn chế lớn với mục tiêu cảnh báo sớm.')
elif recall0 < recall1 - 0.10:
    print(f'- Recall lớp 0 ({recall0:.1%}) thấp hơn rõ rệt recall lớp 1 ({recall1:.1%}). Mô hình cảnh báo rộng và có {fp} cảnh báo nhầm. Recall lớp 1 cao không tự động có nghĩa mô hình tốt.')
else:
    print(f'- Recall của hai lớp tương đối gần nhau (lớp 0: {recall0:.1%}; lớp 1: {recall1:.1%}); vẫn cần xem precision và số lỗi thực tế.')
print(f'- Precision lớp 0 = {precision0:.1%}; Precision lớp 1 = {precision1:.1%}. Với lớp 1, Precision cho biết tỷ lệ cảnh báo đúng trong số các cảnh báo.')
print('- Tập test nhỏ nên các chỉ số có độ bất định; cần đọc chúng cùng với kết quả trung bình/độ lệch chuẩn của CV.')


## 18. Đường Precision–Recall

Đường Precision–Recall mô tả sự đánh đổi giữa Precision và Recall khi ngưỡng phân loại thay đổi. Average Precision tóm tắt chất lượng xếp hạng trên các ngưỡng. Đường trên test được dùng để mô tả hiệu năng cuối cùng; không dùng nó để lựa chọn lại ngưỡng hoặc mô hình.


In [ ]:
precision_curve, recall_curve, _ = precision_recall_curve(y_test, xac_suat_test)
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(recall_curve, precision_curve, label=f"Average Precision = {chi_so_test['Average Precision']:.3f}")
ax.axhline(y_test.mean(), linestyle='--', label=f'Tỷ lệ lớp 1 trong test = {y_test.mean():.3f}')
ax.set_xlabel('Recall lớp 1'); ax.set_ylabel('Precision lớp 1')
ax.set_title('Đường Precision–Recall trên test')
ax.legend(); ax.grid(alpha=.25); plt.tight_layout()
plt.savefig(FIGURES_DIR / '08_duong_precision_recall_test.png', dpi=160); plt.show()


## 19. Phân tích tầm quan trọng của đặc trưng

`Permutation importance` đo mức giảm của Average Precision khi giá trị của từng đặc trưng bị xáo trộn. Nếu việc xáo trộn làm metric giảm nhiều, mô hình đang phụ thuộc tương đối nhiều vào thông tin của đặc trưng đó trong tập đánh giá.

Đây là phân tích hậu kiểm trên test nhằm mô tả hành vi của mô hình đã chọn. Kết quả không chứng minh quan hệ nhân quả và không được dùng để tiếp tục chọn biến rồi báo cáo lại trên cùng test, vì điều đó làm suy yếu tính độc lập của đánh giá cuối cùng.


In [ ]:
try:
    importance = permutation_importance(
        pipeline_chon, X_test[cot_chon], y_test,
        scoring='average_precision', n_repeats=15, random_state=RANDOM_STATE, n_jobs=1
    )
    bang_importance = pd.DataFrame({
        'Đặc trưng': cot_chon,
        'Mức giảm Average Precision trung bình': importance.importances_mean,
        'Độ dao động': importance.importances_std,
    }).sort_values('Mức giảm Average Precision trung bình', ascending=False)
    display(bang_importance.round(4))
    bang_importance.to_csv(OUTPUT_DIR / 'tam_quan_trong_hoan_vi_test_hau_kiem.csv', index=False)
    top_imp = bang_importance.head(12).sort_values('Mức giảm Average Precision trung bình')
    fig, ax = plt.subplots(figsize=(8, max(4, len(top_imp) * .35)))
    ax.barh(top_imp['Đặc trưng'], top_imp['Mức giảm Average Precision trung bình'], xerr=top_imp['Độ dao động'])
    ax.axvline(0, color='black', linewidth=.8)
    ax.set_xlabel('Mức giảm Average Precision khi xáo trộn cột')
    ax.set_title('Permutation importance — phân tích hậu kiểm')
    plt.tight_layout(); plt.savefig(FIGURES_DIR / '09_tam_quan_trong_dac_trung_hau_kiem.png', dpi=160); plt.show()
except Exception as e:
    bang_importance = pd.DataFrame()
    print('Không tạo được permutation importance:', e)

print('Không nên kết luận biến “gây ra” kết quả từ biểu đồ này. Khi nhiều biến tương quan nhau, mức quan trọng của từng biến có thể thay đổi.')


## 20. Lưu mô hình và kết quả thực nghiệm

Mô hình đã chọn, ngưỡng dự đoán, tên tập đặc trưng, định nghĩa nhãn và nguồn dữ liệu được lưu cùng nhau để hỗ trợ tái lập suy luận trên dữ liệu mới. Các bảng CSV, JSON, biểu đồ và báo cáo được lưu tách biệt với dữ liệu nguồn và mã nguồn.


In [ ]:
model_path = MODEL_DIR / 'mo_hinh_canh_bao_hoc_tap.joblib'
artifact = {
    'model': pipeline_chon,
    'threshold': nguong_chon,
    'feature_columns': cot_chon,
    'selected_model': ten_mo_hinh_chon,
    'selected_feature_set': ten_nhom_chon,
    'target_definition': 'at_risk=1 khi G3 < 10; at_risk=0 khi G3 >= 10',
    'data_source': nguon_du_lieu,
}
joblib.dump(artifact, model_path)

ma_tran = {'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp}
summary = {
    **{k: float(v) for k, v in chi_so_test.items()},
    'mo_hinh': ten_mo_hinh_chon,
    'nhom_dac_trung': ten_nhom_chon,
    'so_dac_trung': len(cot_chon),
    'nguong': float(nguong_chon),
    'cv_average_precision_mean': float(phuong_an['average_precision_TB']),
    'cv_average_precision_std': float(phuong_an['average_precision_ĐLC']),
    'cv_f1_class1_mean': float(phuong_an['f1_at_risk_TB']),
    'cv_f1_class1_std': float(phuong_an['f1_at_risk_ĐLC']),
    'ma_tran_nham_lan': ma_tran,
    'nguon_du_lieu': nguon_du_lieu,
    'note': 'Nếu dùng dữ liệu demo, chỉ số dùng để kiểm thử code, không báo cáo như benchmark chính thức.'
}
(OUTPUT_DIR / 'chi_so_test.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

cv_markdown = bang_cv[['Mô hình','Nhóm đặc trưng','Số biến','average_precision_TB','average_precision_ĐLC','f1_at_risk_TB','f1_at_risk_ĐLC','recall_at_risk_TB']].to_markdown(index=False, floatfmt='.3f')
class_markdown = bang_chi_so_lop.round(3).to_markdown()
report_lines = [
    '# Báo cáo mô hình: cảnh báo sớm nguy cơ chưa đạt môn Toán', '',
    '## Giới thiệu dữ liệu',
    f'- Nguồn: `{nguon_du_lieu}`.',
    f'- Số học sinh: {len(data)}; số cột nguồn: {len(df.columns)}.',
    '- Nguồn gốc: UCI Student Performance, môn Toán, hai trường ở Bồ Đào Nha; không đại diện trực tiếp cho học sinh Việt Nam.',
    '- Nhãn lớp 1: G3 < 10 trên thang 0–20; không dùng G1/G2/G3 làm đầu vào.',
    '', '## Thiết kế đánh giá',
    f'- Giữ riêng {len(y_test)} mẫu test ({TEST_SIZE:.0%}).',
    f'- Dùng Stratified {N_SPLITS}-fold CV lặp {N_REPEATS} lần trên train.',
    '- So sánh nhóm học tập cốt lõi 6 biến với toàn bộ 30 biến không phải điểm.',
    '- Xếp hạng theo Average Precision CV, rồi F1 lớp 1; ngưỡng được chọn từ dự đoán OOF trên train.',
    '', '## So sánh kiểm tra chéo', cv_markdown,
    '', '## Đặc trưng liên hệ với nhãn trong train — biến số/thứ bậc',
    bang_tuong_quan.head(10).round(3).to_markdown(index=False),
    '', '## Đặc trưng liên hệ với nhãn trong train — biến phân loại',
    bang_categorical.head(10).round(3).to_markdown(index=False),
    '', '## Mô hình/nhóm đặc trưng được chọn',
    f'- Mô hình: **{ten_mo_hinh_chon}**.',
    f'- Nhóm đặc trưng: **{ten_nhom_chon}** ({len(cot_chon)} biến).',
    f'- Ngưỡng dự đoán từ train OOF: **{nguong_chon:.3f}**.',
    f'- Average Precision CV: {phuong_an["average_precision_TB"]:.3f} ± {phuong_an["average_precision_ĐLC"]:.3f}.',
    f'- F1 lớp 1 CV: {phuong_an["f1_at_risk_TB"]:.3f} ± {phuong_an["f1_at_risk_ĐLC"]:.3f}.',
    '', '## Kết quả test',
    pd.DataFrame([chi_so_test]).round(3).to_markdown(index=False),
    '', '## Chỉ số từng lớp', class_markdown,
    '', '## Insight từ ma trận nhầm lẫn',
    f'- TN={tn}, FP={fp}, FN={fn}, TP={tp}.',
    f'- Recall lớp 0={recall0:.1%}; Recall lớp 1={recall1:.1%}; Precision lớp 1={precision1:.1%}.',
]
if recall1 < recall0 - .10:
    report_lines.append(f'- Recall lớp 1 thấp hơn rõ rệt; bỏ sót {fn} học sinh có nguy cơ — cần xem lại đánh đổi Recall/Precision.')
elif recall0 < recall1 - .10:
    report_lines.append(f'- Recall lớp 0 thấp hơn rõ rệt; mô hình có {fp} cảnh báo nhầm — không thể chỉ dựa vào Recall lớp 1 cao để kết luận tốt.')
else:
    report_lines.append('- Recall hai lớp tương đối gần nhau; vẫn cần xét số cảnh báo nhầm/bỏ sót và quy mô test.')
report_lines += [
    '', '## Giới hạn',
    '- Mẫu nhỏ; kết quả phụ thuộc cách chia mẫu và có độ bất định.',
    '- Mối liên hệ thống kê không chứng minh quan hệ nhân quả.',
    '- Biến nhân khẩu/xã hội có thể mang rủi ro thiên lệch; mô hình chỉ phục vụ học tập/hỗ trợ, không dùng làm căn cứ duy nhất để ra quyết định về học sinh.',
    '- Nếu nguồn là dữ liệu demo tổng hợp, không sử dụng các chỉ số làm benchmark chính thức.',
]
(REPORT_DIR / 'bao_cao_mo_hinh.md').write_text('\n'.join(report_lines), encoding='utf-8')

print('Đã lưu mô hình:', model_path)
print('Đã lưu chỉ số:', OUTPUT_DIR / 'chi_so_test.json')
print('Đã lưu bảng CV:', OUTPUT_DIR / 'so_sanh_kiem_tra_cheo.csv')
print('Đã lưu báo cáo:', REPORT_DIR / 'bao_cao_mo_hinh.md')
print('\nTệp trong outputs:')
for file in sorted(OUTPUT_DIR.iterdir()):
    print('-', file.name)


## 21. Minh họa suy luận trên một quan sát

Một quan sát trong tập test được dùng để minh họa cách đưa dữ liệu qua pipeline đã khớp và tính xác suất ước lượng thuộc lớp 1. Xác suất là đầu ra của mô hình, không phải độ chắc chắn tuyệt đối và không phải kết luận cuối cùng về một cá nhân. Trong ứng dụng thực tế, chỉ được sử dụng những đặc trưng thực sự sẵn có tại thời điểm cảnh báo.


In [ ]:
truong_hop = X_test[cot_chon].iloc[[0]].copy()
xac_suat = float(pipeline_chon.predict_proba(truong_hop)[:, 1][0])
nhan_du_doan = int(xac_suat >= nguong_chon)
print('Thông tin đầu vào:')
display(truong_hop)
print(f'Xác suất ước lượng thuộc nhóm G3 < 10: {xac_suat:.1%}')
print('Nhãn dự đoán:', 'Nhóm cần quan tâm (1)' if nhan_du_doan else 'Không thuộc nhóm cần quan tâm (0)')
print('Không dùng một dự đoán đơn lẻ để gắn nhãn hoặc ra quyết định quan trọng về học sinh.')


## 22. Thảo luận và giới hạn nghiên cứu

Kết quả cần được diễn giải theo ba khía cạnh: (i) hiệu quả trung bình và độ biến động qua kiểm tra chéo; (ii) khác biệt giữa lớp 0 và lớp 1, đặc biệt số trường hợp bỏ sót (FN) và cảnh báo nhầm (FP); (iii) sự khác biệt giữa nhóm đặc trưng cốt lõi và nhóm mở rộng. Một mô hình chỉ có ý nghĩa so với baseline khi cải thiện được các tiêu chí liên quan đến mục tiêu nghiên cứu và kết quả đủ ổn định.

Nghiên cứu có một số giới hạn. Dữ liệu chỉ gồm 395 học sinh từ hai trường tại Bồ Đào Nha; ngưỡng `G3 < 10` là quy ước của bài toán; một số yếu tố giáo dục quan trọng có thể chưa được quan sát; và các mối liên hệ thống kê không chứng minh quan hệ nhân quả. Vì vậy, kết quả chỉ phục vụ mục tiêu học tập và phân tích phương pháp, không dùng làm căn cứ duy nhất để dán nhãn, kỷ luật hoặc ra quyết định quan trọng về học sinh.

Các kết luận định lượng phải lấy trực tiếp từ lần chạy trên CSV UCI chính thức. Kết quả tạo ra từ dữ liệu minh họa chỉ xác nhận khả năng thực thi của mã, không phải bằng chứng thực nghiệm về hiệu quả mô hình.
